# Synthetic Data Generator
### Step 8 : Kafka Consumer Adapter

In [ ]:
import os

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
)

from utils.synthetic.pipeline.kafka_consumer_adapter import (
    run_kafka_consumer_to_postgres_once, 
    run_kafka_consumer_to_postgres_loop,
)

from utils.core.env_helpers import (
    env_bool,
    env_float,
    env_int,
    env_optional_int,
    env_str,
)



In [ ]:
SCHEMA = env_str(
    "CAPSTONE_SCHEMA",
    "capstone",
    aliases=("CONSUMER_SCHEMA",),
)

DATASET_ID = env_str(
    "SYNTHETIC_DATASET_ID",
    "pump_synthetic_v1",
    aliases=("DATASET_ID",),
)
RUN_ID = env_str(
    "SYNTHETIC_RUN_ID",
    "synthetic_run_001",
    aliases=("RUN_ID",),
)

NUMBER_OF_SENSORS = env_int("SYNTHETIC_SENSOR_COUNT", 52)

OBSERVATION_BATCH_SIZE = env_int(
    "OBSERVATION_BATCH_SIZE",
    500,
    aliases=("SYNTHETIC_OBSERVATION_BATCH_SIZE",),
)

TOPIC = env_str(
    "SYNTHETIC_KAFKA_TOPIC",
    "pump.telemetry.synthetic",
    aliases=("KAFKA_TOPIC",),
)

CONSUMER_DESTINATION_TABLE_NAME = env_str(
    "SYNTHETIC_CONSUMED_MESSAGES_TABLE",
    "synthetic_sensor_messages_consumed_stage",
    aliases=("CONSUMER_TARGET_TABLE",),
)

CONSUMER_GROUP_ID = env_str(
    "SYNTHETIC_CONSUMER_GROUP_ID",
    "synthetic-telemetry-consumer-group",
    aliases=("KAFKA_CONSUMER_GROUP_ID", "CONSUMER_GROUP_ID"),
)

CONSUMER_WORKER_ID = env_str(
    "SYNTHETIC_CONSUMER_WORKER_ID",
    "consumer_worker_001",
    aliases=("CONSUMER_WORKER_ID",),
)

DEFAULT_CONSUMER_BATCH_SIZE = OBSERVATION_BATCH_SIZE * NUMBER_OF_SENSORS

CONSUMER_BATCH_SIZE = env_int(
    "CONSUMER_BATCH_SIZE",
    DEFAULT_CONSUMER_BATCH_SIZE,
)

CONSUMER_POLL_TIMEOUT_SECONDS = env_float(
    "CONSUMER_POLL_TIMEOUT_SECONDS",
    1.0,
)

AUTO_OFFSET_RESET = env_str(
    "CONSUMER_AUTO_OFFSET_RESET",
    "earliest",
    aliases=("SYNTHETIC_AUTO_OFFSET_RESET",),
)

MAX_BATCHES = env_optional_int(
    "CONSUMER_MAX_BATCHES_LIMIT",
    default=100000,
)

IDLE_SLEEP_SECONDS = env_float(
    "CONSUMER_IDLE_SLEEP_SECONDS",
    0.0,
)

STOP_ON_EMPTY_FLAG = env_bool(
    "CONSUMER_STOP_ON_EMPTY",
    True,
)

print("Kafka consumer config")
print("schema:", SCHEMA)
print("dataset id:", DATASET_ID)
print("run id:", RUN_ID)
print("topic:", TOPIC)
print("target table:", CONSUMER_DESTINATION_TABLE_NAME)
print("consumer group id:", CONSUMER_GROUP_ID)
print("consumer worker id:", CONSUMER_WORKER_ID)
print("observation batch size:", OBSERVATION_BATCH_SIZE)
print("message batch size:", CONSUMER_BATCH_SIZE)
print("max batches:", MAX_BATCHES)
print("stop on empty:", STOP_ON_EMPTY_FLAG)

---

In [ ]:
engine = get_engine_from_env()

---

#### Single Batch

In [ ]:
result = run_kafka_consumer_to_postgres_once(
    engine=engine,
    topic=TOPIC,
    schema=SCHEMA,
    table_name=CONSUMER_DESTINATION_TABLE_NAME,
    max_messages=CONSUMER_BATCH_SIZE,
    poll_timeout_seconds=CONSUMER_POLL_TIMEOUT_SECONDS,
    consumer_group_id=CONSUMER_GROUP_ID,
    consumer_worker_id=CONSUMER_WORKER_ID,
    auto_offset_reset=AUTO_OFFSET_RESET,
)

display(result)

----

#### Loop Batches

In [ ]:
results = run_kafka_consumer_to_postgres_loop(
    engine=engine,
    topic=TOPIC,
    schema=SCHEMA,
    table_name=CONSUMER_DESTINATION_TABLE_NAME,
    max_messages=CONSUMER_BATCH_SIZE,
    poll_timeout_seconds=CONSUMER_POLL_TIMEOUT_SECONDS,
    consumer_group_id=CONSUMER_GROUP_ID,
    consumer_worker_id=CONSUMER_WORKER_ID,
    auto_offset_reset=AUTO_OFFSET_RESET,
    max_batches=MAX_BATCHES,
    idle_sleep_seconds=IDLE_SLEEP_SECONDS,
    stop_on_empty=STOP_ON_EMPTY_FLAG,
)

display(results)

----

In [ ]:
validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT observation_index) AS distinct_observation_count,
        COUNT(DISTINCT kafka_topic || ':' || kafka_partition || ':' || kafka_offset) AS distinct_kafka_messages,
        MIN(consumer_received_at) AS min_consumer_received_at,
        MAX(consumer_received_at) AS max_consumer_received_at
    FROM {SCHEMA}.{CONSUMER_DESTINATION_TABLE_NAME}
    """
)

display(validation_dataframe)

----

In [ ]:
# Dispose Engine
engine.dispose()